# Rover on Rough Terrain — Planning Challenge (C++ edition, 30 min)

A point rover must drive from `start` to `goal` across rough 2D terrain.

**You have:**
- `field` `(H,W,3)` — terrain channels: **rock** (0/1, lethal), **mud** (0..1),
  **slope** (0..1). `x = column`, `y = row`.
- `demos` — a few expert `(x,y)` paths, each between its *own* start/goal (not
  yours). No single one is the answer; they reveal which terrain is costly.
- `astar_plan(w, cost_map)` — a **provided** planner (`planner_p1.hpp`). Give it
  a cost map, get a near-optimal path. You do **not** write a planner.

**Your one task:** write `cost_map_learned` (in `learn_p1.hpp`) → a cost map
that, routed through `astar_plan`, yields a path as terrain-cheap as the experts'.
The provided `cost_map_baseline` ignores mud, so its path FAILs — improve on it.
You only `%%writefile learn_p1.hpp`.

### Setup

In [ ]:
import os, subprocess, struct
import numpy as np
if not os.path.exists('field_p1.npy'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field_p1.npy').astype(np.float32)        # (H, W, 3)
demos = np.load('demos_p1.npy', allow_pickle=True)        # object array of (T,2)
meta  = np.load('meta_p1.npz')
H, W = field.shape[:2]

with open('world_p1.bin', 'wb') as f:
    f.write(struct.pack('<ii', H, W))
    f.write(np.ascontiguousarray(field, np.float32).tobytes())
    f.write(np.ascontiguousarray(meta['true_cost'], np.float64).tobytes())
    f.write(np.asarray(meta['start'], np.float64).tobytes())
    f.write(np.asarray(meta['goal'],  np.float64).tobytes())
    f.write(struct.pack('<ddd', float(meta['optimal_cost']), float(meta['tol']),
                        float(meta['rock_sentinel'])))
    f.write(struct.pack('<i', len(demos)))
    for d in demos:
        d = np.asarray(d, np.float64)
        f.write(struct.pack('<i', d.shape[0]))
        f.write(np.ascontiguousarray(d, np.float64).tobytes())

import sys
sys.path.insert(0, os.getcwd())
from viz_p1 import show_cpp, build_and_run

print('g++ version:',
      subprocess.run(['g++', '--version'], capture_output=True, text=True).stdout.splitlines()[0])
print(f'packed world_p1.bin: field {field.shape} | demos {len(demos)} '
      f'| start {meta["start"]} goal {meta["goal"]}')
# Compile-check the shipped driver that needs no candidate code, so a bad clone
# or toolchain surfaces here rather than inside the TODO.
_chk = subprocess.run(['g++', '-O2', '-std=c++17', 'main_show_p1.cpp', '-o', '_chk_p1'],
                      capture_output=True, text=True)
print('toolchain + shipped sources:', _chk.stderr.strip() or 'OK')

### API (all prebuilt — do not edit; ships in the cloned repo)

Everything is passed the `World w`:
- `w.H`, `w.W`; `w.start`, `w.goal` (`Vec2{x,y}`, `x=col, y=row`).
- `w.demos` — `std::vector<Path>` (`Path = std::vector<Vec2>`), expert polylines
  with their *own* endpoints.
- `w.fld(r,c,ch) -> float` — terrain at row `r`, col `c`, channel
  `ROCK`/`MUD`/`SLOPE` (=0/1/2). Rock is 0/1 & lethal; mud/slope ∈ 0..1.

**Cost maps** are `std::vector<double>`, length `H*W`, row-major: `cost_map[r*W + c]`.

**Call these:** `astar_plan(w, cost_map) -> Path` (provided planner),
`cost_map_baseline(w)`, `path_cost(w, p, cost_map)`, `evaluate(w, p, cost_map)`.

You implement ONLY `cost_map_learned(const World& w)` in `learn_p1.hpp`.

### Terrain + expert demos (gray)

In [ ]:
build_and_run('main_show_p1.cpp', [('initial_p1.bin', 'terrain (true cost) + expert demos (gray)')])

### Baseline run &mdash; expect `FAIL`

In [ ]:
build_and_run('main_run_baseline_p1.cpp', [('run_baseline_p1.bin', 'baseline cost: planned path')])

## TODO &mdash; `cost_map_learned` in `learn_p1.hpp` (see the cell)

In [ ]:
%%writefile learn_p1.hpp
// ---- TODO: learn a (H,W) cost map from the demos -------------------------
// Make terrain the experts drive through cheap, and what they detour around
// costly, so astar_plan routes like them. You only need the RANKING right, not
// the true cost numerically. Keep rocks at ROCK_PENALTY.
// (e.g. count w.demos visitation, or weight up mud/slope -- your call.)
#pragma once
#include "baseline_p1.hpp"
#include "terrain_p1.hpp"

inline std::vector<double> cost_map_learned(const World& w) {
    // TODO: use w.demos. (Falls back to the wrong baseline for now.)
    return cost_map_baseline(w);
}

### Run with your learned cost &mdash; goal: `PASS`

In [ ]:
build_and_run('main_run_p1.cpp', [('run_cost_p1.bin', 'learned: planned path over learned cost map'), ('run_terrain_p1.bin', 'learned: planned path over terrain')])

### Discussion (optional — no code needed)

Be ready to talk through: where your learned cost generalizes badly; A* vs.
sampling vs. trajectory-opt and what each guarantees; the planner's Euclidean
heuristic (admissible here?); 8- vs 4-connectivity and resolution tradeoffs;
`float` vs `double` / cache locality in C++; a safety margin around rocks;
replanning in a 50 Hz loop.